# 10 · Rust/WASM spatial computation

**Question:** When does a compiled spatial kernel help, and what does boundary overhead cost?

**Runtime:** Python / Pyodide in JupyterLite. Choose **Run → Run All Cells**.
Each notebook is independent; start with a fresh kernel. Initial runtime/package downloads need internet.
The initial baseline uses **synthetic educational fixtures** near a Gaborone-area anchor.
Extended investigations explicitly identify bundled real OSM, Sentinel or terrain extracts where used;
these do not validate the synthetic population, mobility, exposure or scenario assumptions.

**Astra experiment:** Ask Astra/Codex to optimize the Rust kernel without changing its formula, then compare numerical parity and median runtime.
Run the numerical baseline first, then give Astra the exported evidence and the prompt below.
The baseline does not call or run GPT-6 Astra. No API key is needed.


In [ ]:
import sys, json, math, time
from pathlib import Path
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["numpy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from twin import *
style()
print("Runtime:", sys.platform, "| data: synthetic | seed: 42")


## A small Rust export called from Pyodide
`rust/src/lib.rs` sums an inverse-quadratic distance weight over deterministic synthetic points.
GitHub Actions compiles it to `content/assets/twin_kernel.wasm` before building the site.
In a browser, Pyodide calls JavaScript WebAssembly through its foreign-function interface.
Desktop execution uses Wasmtime when the compiled asset is available. A missing asset is reported explicitly.
The benchmark includes the call boundary but excludes module initialization. It does not establish a general Rust speedup.


In [ ]:
def reference(n,x,y,decay):
    return sum(1/(1+(((i*37%2000)-x)**2+((i*71%2000)-y)**2)/decay**2) for i in range(n))
def vectorized(n,x,y,decay):
    i=np.arange(n); return float(np.sum(1/(1+((i*37%2000-x)**2+(i*71%2000-y)**2)/decay**2)))
asset=Path('assets/twin_kernel.wasm')
wasm_fn=None
if sys.platform=='emscripten':
    import js
    from pyodide.ffi import to_js
    if asset.exists():
        compiled=await js.WebAssembly.compile(to_js(asset.read_bytes()))
        instance=await js.WebAssembly.instantiate(compiled)
        wasm_fn=instance.exports.exposure_sum
elif asset.exists():
    try:
        import wasmtime
        engine=wasmtime.Engine(); store=wasmtime.Store(engine)
        module=wasmtime.Module.from_file(engine,str(asset)); instance=wasmtime.Instance(store,module,[])
        function=instance.exports(store)['exposure_sum']
        wasm_fn=lambda *args:function(store,*args)
    except ImportError:
        print('Desktop Wasmtime is not installed; use the published JupyterLite site.')
print('Rust/WASM loaded' if wasm_fn else 'Rust/WASM NOT loaded. Build it using docs/GUIDE.md; Python comparisons still run.')


In [ ]:
n=20000; args=(n,1000.,900.,200.)
expected=reference(*args)
assert np.isclose(vectorized(*args),expected,rtol=1e-11)
functions={'Python loop':reference,'NumPy':vectorized}
if wasm_fn:
    assert np.isclose(wasm_fn(*args),expected,rtol=1e-11)
    assert wasm_fn(0,0.,0.,100.)==0.
    functions['Rust/WASM']=wasm_fn
timings={}
for name,fn in functions.items():
    fn(*args); samples=[]
    for _ in range(5):
        start=time.perf_counter(); value=fn(*args); samples.append((time.perf_counter()-start)*1000)
    timings[name]=float(np.median(samples))
fig,ax=plt.subplots(); ax.bar(timings.keys(),timings.values(),color=['#087f8c','#f2a541','#536dfe'][:len(timings)])
ax.set(ylabel='Median milliseconds (5 runs)',title=f'Same formula, {n:,} points, this device'); plt.show()
export_json('10_benchmark.json',dict(synthetic=True,wasm_loaded=wasm_fn is not None,n=n,median_ms=timings,reference=expected))


## Teaching lab: Optimize without changing meaning
**Before using AI:** write a prediction, run the numerical baseline, and explain one surprising result.

**Astra evidence:** send only rust/src/lib.rs and 10_benchmark.json. Use a fresh conversation and record the exact model,
date, interface, tools and prompt. A task with a supplied result tests interpretation, not independent calculation.

**Copyable Astra prompt:**
> Propose one optimization, identify its expected crossover costs, and state the parity checks needed before comparing timings.

**Implementation brief — review the runnable extension below:**
> Benchmark multiple point counts with warm-up and repeated measurements; preserve the arithmetic formula. Edit the authoring source in `scripts/make_notebooks.py` (or
> `scripts/teaching_labs.py` for labs 11–13), regenerate the notebooks and preserve browser compatibility.
> Use NumPy, Matplotlib and the standard library. Include a small reference case that can be checked by hand.

**Acceptance evidence:** Python/NumPy/WASM agree within declared tolerance including zero points; module initialization and call overhead are labeled.

The extended investigation later in this notebook implements this line of work with maps and checks.
Use the brief to review the implementation or ask Codex for a further controlled variation.

**Share back:** save your original prediction, changed parameter, result, model response and one limitation.
Download the edited notebook and exports. If you have no model access, exchange explanations with a partner
and evaluate them with the same criteria; do not label that activity as an Astra trial.


## Extended investigation: Batch spatial queries and check a grid index

Batch 32 queries into one NumPy operation, compare with repeated NumPy and WASM calls, and map a separate radius-query grid index. The original inverse-quadratic kernel has infinite support: a radius index cannot discard its far-field contributions while claiming exact parity. The index example therefore answers an explicitly different question—points within 200 m—and compares that answer to brute force.

**Read the concept map first:** identify the input, method, evidence and limit. Predict the spatial pattern before running the comparison.


In [ ]:
from extension_tools import concept_map, export_investigation
concept_map('Batch spatial queries and check a grid index', ['Point set + many query sites', 'Batched NumPy / WASM calls', 'Parity + crossover chart', 'Finite radius ≠ infinite kernel'])
plt.show()


In [ ]:
from extension_tools import radius_index, radius_query
ex_queries=np.column_stack([np.linspace(100,1900,32),1000+600*np.sin(np.linspace(0,2*np.pi,32))])
def ex_batch(count,queries,decay=200.):
    i=np.arange(count); points=np.column_stack([i*37%2000,i*71%2000]); d2=((queries[:,None,:]-points[None,:,:])**2).sum(axis=2)
    return (1/(1+d2/decay**2)).sum(axis=1)
ex_sizes=[100,1000,5000]; ex_timings={'Batched NumPy':[],'Repeated NumPy':[]}
if wasm_fn: ex_timings['Repeated WASM']=[]
for count in ex_sizes:
    expected=ex_batch(count,ex_queries)
    functions={'Batched NumPy':lambda:ex_batch(count,ex_queries),'Repeated NumPy':lambda:np.array([vectorized(count,x,y,200.) for x,y in ex_queries])}
    if wasm_fn: functions['Repeated WASM']=lambda:np.array([wasm_fn(count,float(x),float(y),200.) for x,y in ex_queries])
    for name,fn in functions.items():
        assert np.allclose(fn(),expected,rtol=1e-11); samples=[]
        for repeat in range(3): start=time.perf_counter(); fn(); samples.append((time.perf_counter()-start)*1000)
        ex_timings[name].append(float(np.median(samples)))
assert np.array_equal(ex_batch(0,ex_queries),np.zeros(32))
i=np.arange(5000); ex_points=np.column_stack([i*37%2000,i*71%2000]); ex_buckets=radius_index(ex_points,200.)
ex_center=ex_queries[10]; ex_hits=radius_query(ex_points,ex_buckets,ex_center,200.,200.)
ex_brute=np.flatnonzero(np.linalg.norm(ex_points-ex_center,axis=1)<=200.)
assert np.array_equal(ex_hits,ex_brute)
fig,axes=plt.subplots(1,3,figsize=(14,4))
axes[0].scatter(*ex_points.T,s=2,c='#bdcbd0'); axes[0].scatter(*ex_points[ex_hits].T,s=8,c='#087f8c'); axes[0].add_patch(plt.Circle(ex_center,200,fill=False,color='#c44e52')); map_axes(axes[0],'Exact finite-radius index query')
for name,values in ex_timings.items(): axes[1].plot(ex_sizes,values,marker='o',label=name)
axes[1].set(xscale='log',yscale='log',xlabel='Point count',ylabel='Median ms, 3 repeats',title='32 queries on this device'); axes[1].legend(fontsize=8)
dots=axes[2].scatter(*ex_queries.T,c=ex_batch(5000,ex_queries),s=65,cmap='magma'); map_axes(axes[2],'Full-support kernel at query sites'); fig.colorbar(dots,ax=axes[2],label='Kernel sum')
fig.tight_layout(); extension_figure=fig
extension_evidence=dict(timings_ms=ex_timings,point_counts=ex_sizes,query_count=32,index_count=len(ex_hits),brute_force_count=len(ex_brute),kernel='1/(1+distance_squared/200**2)',radius_m=200)
extension_expected=dict(index_matches_brute_force=True,truncation_preserves_full_kernel=False)
extension_task='Return index_matches_brute_force and truncation_preserves_full_kernel. Distinguish the finite-radius count from the full inverse-quadratic sum.'


In [ ]:
extension_prompt,extension_reference=export_investigation('10',extension_evidence,extension_expected,extension_task,extension_figure)
plt.show()
print('Send only 10_extension_prompt.json and optionally 10_extension_map.png to Astra. Keep the reference in this kernel.')


### Ask, check and explain the spatial result
Download `10_extension_prompt.json` and `10_extension_map.png`. Send the evidence in a fresh Astra
session with the exact task and output fields. For a separate image-only experiment, supply only the image
and task wording, record that condition, and allow clarification if the image does not contain enough information.
Never send the reference or this complete notebook for a blind trial.

Paste the unedited response below or upload a UTF-8 response file. The default is deliberately empty:
**no model trial has run**. After recording a real answer, set `extension_source='MODEL'` and complete the metadata.
The numeric tolerance is 0.01 in each requested field's declared units; labels and booleans require exact agreement.
Keep that rule fixed before examining a response. Inspect the explanation separately for unsupported claims.

**Visual reflection:** point to one map pattern, cite the numerical check that supports it, and name one thing
the visual cannot establish. Re-run one parameter change and preserve both maps. The implementation above
is available for inspection and modification; it is not a measured claim about model accuracy.


In [ ]:
from experiments import grade_answer
extension_source='NOT RUN'
extension_model=''; extension_date=''; extension_interface=''; extension_condition='text-and-map'
extension_response_file=''; extension_response_text=''
if extension_response_file:
    extension_response_text=Path(extension_response_file).read_text(encoding='utf-8')
if extension_source=='MODEL':
    assert extension_model and extension_date and extension_interface and extension_response_text, 'Record the actual model, date, interface and response'
extension_tolerances={key:.01 for key,value in extension_reference.items() if type(value) in (int,float)}
extension_grade=grade_answer(extension_response_text,extension_reference,extension_tolerances) if extension_response_text else dict(status='NOT RUN')
display(extension_grade)
export_json('10_extension_trial.json',dict(source=extension_source,model=extension_model,date=extension_date,
    interface=extension_interface,condition=extension_condition,prompt=extension_prompt,response_text=extension_response_text,
    tolerances=extension_tolerances,reference=extension_reference,grade=extension_grade))


## Save the investigation
Download the edited notebook, evidence, map and trial record from `exports/`. Browser storage does not
commit changes to GitHub. Keep data attribution and the distinction between observed data, synthetic
assumptions and actual model responses when sharing. The extended code and its checks above are part of
this notebook; use them as the starting point for your next controlled comparison.
